# wp1pt4pt1g — set up MSA runs for the sites that are ready

Reads the completed-site summary produced by the record-availability audit
(`data_processed/06_gm_selection/complete_site_im_sets.csv`) and, for every
**ready** (site, conditioning-IM) unit, wires up the Multiple-Stripe-Analysis
(MSA) run files in the **existing** per-site `sdof_param/` and `mdof/` analysis
folders (built by `wp1pt4pt1f`), then writes **4 batch launchers** — SDOF and
MDOF kept separate, each split by IM.

This notebook does **not** design structures or run modal analyses. It only adds
the MSA files (`run_msa_site.py`, the max-parallel coordinator + worker,
injection/recorder helpers, and one `config_msa_{IM}.py` + stripe-pickle folder
per IM). If a system folder or its `structural_model.py` is missing, that
(site, system) is **warned and skipped** (run `wp1pt4pt1f` first).

Maximum parallelisation: `run_msa_site.py` -> `run_batch_msa_per_stripe_record.py`
(dispatches every `(stripe, record)` pair at once) -> `run_msa_per_record.py` worker.


In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [2]:
import re
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_file,
    copy_analysis_config,
    configure_batch_run_file,
)

cfg = config.load_config()

# --- parameters (match wp1pt4pt1f) ---
DEST_ROOT = cfg["analysis_data"]["site_influence_investigation"]
GM_SETS = ["AvgSA_03", "AvgSA_06"]
STRIPE_ORDER_ASCENDING = True
MAX_N_RECORDS = None                 # cap records per stripe (None = all)

# system folder name -> EDP recorder template key
SYSTEMS = {
    "sdof_param": "ida_process_recorder_x_displacement",   # x-displacement EDP
    "mdof": "ida_process_recorder_roof_drift",             # roof-drift EDP
}

BATCH_ROOT = Path(cfg["scripts"]["wp1pt4pt1_batch_run"])
print(f"DEST_ROOT = {DEST_ROOT}")

DEST_ROOT = D:\04_site_influence_investigation


## 1. Read the ready list

`ready[site] = {ims...}` — the (site, IM) units flagged complete by the
availability audit.

In [3]:
ready_csv = Path(cfg["proc_data"]["gm_selection"]) / "complete_site_im_sets.csv"
ready_df = pd.read_csv(ready_csv)

ready: dict[int, set[str]] = {}
for _, row in ready_df.iterrows():
    ready.setdefault(int(row["site"]), set()).add(str(row["im"]))

print(f"{len(ready)} sites with >=1 ready IM  (from {ready_csv.name})")
for g in GM_SETS:
    n = sum(1 for ims in ready.values() if g in ims)
    print(f"  {g}: {n} ready sites")

34 sites with >=1 ready IM  (from complete_site_im_sets.csv)
  AvgSA_03: 32 ready sites
  AvgSA_06: 33 ready sites


## 2. Discover the per-site stripe pickles

Same discovery as `wp1pt4pt1f`: `site_{ii}__stripe_iml_{iml}__gm_selection.pickle`
under each GM set's record-selection folder (the `iml` token is zero-padded /
`pt`-encoded; only the site index is needed to group a site's stripes).

In [ ]:
# Stripe pickles are named site_{site}__stripe_iml_{iml}__gm_selection.pickle
# (iml zero-padded / pt-encoded). Only the site index is needed here to group a
# site's stripes; the iml token just has to match.
STRIPE_RE = re.compile(r"site_(\d+)__stripe_iml_[0-9]+pt[0-9]+__gm_selection\.pickle$")


def discover_site_stripes(gm_set: str) -> dict[int, list[Path]]:
    """Return {site_idx: [stripe pickle paths]} for a GM set."""
    folder = Path(cfg["results"][f"{gm_set}_record_selection"])
    out: dict[int, list[Path]] = {}
    for f in sorted(folder.iterdir()):
        m = STRIPE_RE.match(f.name)
        if m:
            out.setdefault(int(m.group(1)), []).append(f)
    return out


site_stripes = {g: discover_site_stripes(g) for g in GM_SETS}
for g in GM_SETS:
    print(f"{g}: stripe pickles found for {len(site_stripes[g])} sites")

## 3. `add_msa_files` helper

Copies the MSA run files into a folder and creates one `config_msa_{IM}.py` +
`{IM}/` stripe-pickle subfolder per ready IM. Unlike the `wp1pt4pt1f` version this
also copies the **`run_msa_per_record.py`** worker (required by the max-parallel
coordinator) and only writes configs for the IMs passed in `gm_sets`.

In [ ]:
def _stripe_pickles_for(site_idx: int, gm_set: str) -> list[Path]:
    return site_stripes[gm_set].get(site_idx, [])


def add_msa_files(folder: Path, site_idx: int, recorder_template_key: str,
                  gm_sets: list[str]) -> None:
    """Add MSA run/coordinator/worker/injection/recorder files + one config and
    stripe-pickle subfolder per GM set in `gm_sets`."""
    copy_file(cfg["templates"]["run_msa_site"], folder / "run_msa_site.py")
    copy_file(cfg["templates"]["run_batch_msa_per_stripe_record"],
              folder / "run_batch_msa_per_stripe_record.py")
    copy_file(cfg["templates"]["run_msa_per_record"],
              folder / "run_msa_per_record.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"][recorder_template_key],
              folder / "msa_process_recorders.py")

    record_src = Path(cfg["proc_data"]["gm_records"]).as_posix()
    for gm_set in gm_sets:
        gm_dir = folder / gm_set
        gm_dir.mkdir(parents=True, exist_ok=True)
        # clear any previously-copied stripe pickles first so stale / old-scheme
        # (int-indexed) names don't linger alongside the current iml-named set
        # (find_stripe_pickles globs all *stripe*gm_selection* in the folder).
        for old in gm_dir.glob("*__stripe_*__gm_selection.pickle"):
            old.unlink()
        for src in _stripe_pickles_for(site_idx, gm_set):
            copy_file(src, gm_dir / src.name)
        copy_analysis_config(
            cfg["templates"]["config_msa"],
            folder / f"config_msa_{gm_set}.py",
            results_folder_name=f"msa_{gm_set}",
            gm_selection_src_str=gm_dir.as_posix(),
            record_src_str=record_src,
            stripe_order_ascending=STRIPE_ORDER_ASCENDING,
            max_n_records=MAX_N_RECORDS,
        )

## 4. Add MSA files to the existing SDOF & MDOF folders

For every ready site and each system, add the MSA files to the folder built by
`wp1pt4pt1f`. Missing folders / structural models are warned and skipped (and are
excluded from the batch files).

In [6]:
# system -> {site_idx: folder}, only sites successfully set up (model present)
system_folders: dict[str, dict[int, Path]] = {s: {} for s in SYSTEMS}
skipped: list[tuple[int, str]] = []

for site_idx in sorted(ready):
    gm_sets = sorted(ready[site_idx] & set(GM_SETS))
    if not gm_sets:
        continue
    for system, recorder_key in SYSTEMS.items():
        folder = DEST_ROOT / f"site_{site_idx}" / system
        if not (folder / "structural_model.py").exists():
            print(f"WARNING: skipping site {site_idx} [{system}]: "
                  f"{folder / 'structural_model.py'} not found (run wp1pt4pt1f first)")
            skipped.append((site_idx, system))
            continue
        add_msa_files(folder, site_idx, recorder_key, gm_sets)
        system_folders[system][site_idx] = folder

for system in SYSTEMS:
    print(f"{system}: MSA files added for {len(system_folders[system])} sites")
if skipped:
    print(f"skipped {len(skipped)} (site, system) pair(s): {skipped}")

sdof_param: MSA files added for 34 sites
mdof: MSA files added for 34 sites


## 5. Write the 4 batch launchers

One batch file per (system, IM). Each lists a job per ready site: its
`run_msa_site.py` + the matching `config_msa_{IM}.py`. `configure_batch_run_file`
serialises these into the runnable `scripts_and_configs` launcher (one console
window per job).

In [7]:
BATCH_ROOT.mkdir(parents=True, exist_ok=True)

for system in SYSTEMS:
    for gm_set in GM_SETS:
        jobs = [
            {
                "script": folder / "run_msa_site.py",
                "config": [folder / f"config_msa_{gm_set}.py"],
                "name": [f"site_{site_idx}_{system}_msa_{gm_set}"],
            }
            for site_idx, folder in sorted(system_folders[system].items())
            if gm_set in ready[site_idx]
        ]
        batch_dst = BATCH_ROOT / f"site_{system}_msa_{gm_set}.py"
        configure_batch_run_file(cfg["templates"]["batch_run"], batch_dst, jobs)
        print(f"wrote {batch_dst.name}: {len(jobs)} jobs")

wrote site_sdof_param_msa_AvgSA_03.py: 32 jobs
wrote site_sdof_param_msa_AvgSA_06.py: 33 jobs
wrote site_mdof_msa_AvgSA_03.py: 32 jobs
wrote site_mdof_msa_AvgSA_06.py: 33 jobs
